# Pandas

## Advanced

In [222]:
import numpy as np
import pandas as pd

### Useful features

#### Operations on dataframe

##### shift

In [223]:
# shift function moves the values up or down by a specified number.
s = pd.Series(np.arange(5)).shift()    # by default down values by 1.
print(s)
print()

s = s.shift(-1)
print(s)

0    NaN
1    0.0
2    1.0
3    2.0
4    3.0
dtype: float64

0    0.0
1    1.0
2    2.0
3    3.0
4    NaN
dtype: float64


#### Operations with time

##### Basic creation of time range

In [224]:
# Using date_range
rng = pd.date_range('2025-01-15', periods=5)
print(rng, '\n')

# Using to_datetime
rng = pd.to_datetime([
    '2025-01-15',
    '2025-01-19',
    '2025-03-10',
    '2025-02-16',
    '2025-01-31'
])
print(rng)

DatetimeIndex(['2025-01-15', '2025-01-16', '2025-01-17', '2025-01-18',
               '2025-01-19'],
              dtype='datetime64[us]', freq='D') 

DatetimeIndex(['2025-01-15', '2025-01-19', '2025-03-10', '2025-02-16',
               '2025-01-31'],
              dtype='datetime64[us]', freq=None)


##### freq

In [225]:
# by default = D means 1 day gap
# 2D means 2 days gap
print(pd.date_range('2025-01-15', periods=5, freq='2D'), '\n')

# W for week, ME for month-end, MS for month-start, QE for quarter-end, YE for year-end, h for hour, min for minute, s for second

print(pd.date_range('2025-01-15', periods=5, freq='MS'), '\n')
print(pd.date_range('2025-01-15', periods=5, freq='QE'), '\n')
print(pd.date_range('2025-01-15', periods=5, freq='min'), '\n')

DatetimeIndex(['2025-01-15', '2025-01-17', '2025-01-19', '2025-01-21',
               '2025-01-23'],
              dtype='datetime64[us]', freq='2D') 

DatetimeIndex(['2025-02-01', '2025-03-01', '2025-04-01', '2025-05-01',
               '2025-06-01'],
              dtype='datetime64[us]', freq='MS') 

DatetimeIndex(['2025-03-31', '2025-06-30', '2025-09-30', '2025-12-31',
               '2026-03-31'],
              dtype='datetime64[us]', freq='QE-DEC') 

DatetimeIndex(['2025-01-15 00:00:00', '2025-01-15 00:01:00',
               '2025-01-15 00:02:00', '2025-01-15 00:03:00',
               '2025-01-15 00:04:00'],
              dtype='datetime64[us]', freq='min') 



##### resample

In [226]:
# resample is used to convert timestamped data into a new frequency.
# it works in 2 steps - creates new frequency and then applies aggregation
rng = pd.date_range('2025-03-15', periods=12, freq='5D')
ts = pd.Series(np.random.randint(1,101,12), index=rng)
print(ts[:4])

ts = ts.resample('MS').sum()
ts

2025-03-15    90
2025-03-20    64
2025-03-25    24
2025-03-30    71
Freq: 5D, dtype: int32


2025-03-01    249
2025-04-01    252
2025-05-01    123
Freq: MS, dtype: int32

##### tz_localize and tz_convert

First hh:mm:ss shows the time and second hh:mm shows UTC offset for the timezone.

In [227]:
# tz_localize gives a timezone to a time series.
ts2 = ts.tz_localize('UTC')
ts2

2025-03-01 00:00:00+00:00    249
2025-04-01 00:00:00+00:00    252
2025-05-01 00:00:00+00:00    123
Freq: MS, dtype: int32

In [228]:
# tz_convert converts the timezone to another one and shows information of moments related to older one.
ts3 = ts2.tz_convert('Asia/Kolkata')
ts3

2025-03-01 05:30:00+05:30    249
2025-04-01 05:30:00+05:30    252
2025-05-01 05:30:00+05:30    123
dtype: int32

##### BusinessDay

In [229]:
from pandas.tseries.offsets import BusinessDay
from pandas.tseries.offsets import Day

# BusinessDay skipps the weekends (saturday, sunday) and checks only for working days.
# B in freq of date_range uses frequency for Business Days.
date = pd.Timestamp('2025-01-17')
print(date, date.day_name())

print(date + pd.offsets.Day(1), (date + Day(1)).day_name())
print(date + pd.offsets.BusinessDay(), (date + BusinessDay()).day_name())

2025-01-17 00:00:00 Friday
2025-01-18 00:00:00 Saturday
2025-01-20 00:00:00 Monday


In [230]:
# BusinessDay with series
rng = dates = pd.Series(pd.to_datetime([
    "2025-03-06",
    "2025-03-07",
    "2025-03-08",
    "2025-03-09"
]))
dates + BusinessDay()

0   2025-03-07
1   2025-03-10
2   2025-03-10
3   2025-03-10
dtype: datetime64[us]

#### Categoricals

##### Creating a categorical

In [231]:
df = pd.DataFrame({
    'id': np.arange(1,5),
    'grade': ['A', 'B', 'A', 'C']
})
print(df['grade'], '\n')

# Using astype
df['grade'] = df['grade'].astype('category')
print(df['grade'], '\n')

# Using categorical
df['grade'] = pd.Categorical(['A', 'B', 'A', 'C'])
print(df['grade'])

0    A
1    B
2    A
3    C
Name: grade, dtype: str 

0    A
1    B
2    A
3    C
Name: grade, dtype: category
Categories (3, str): ['A', 'B', 'C'] 

0    A
1    B
2    A
3    C
Name: grade, dtype: category
Categories (3, str): ['A', 'B', 'C']


##### Accessing the information of categorical

In [232]:
df['grade'].dtype

CategoricalDtype(categories=['A', 'B', 'C'], ordered=False, categories_dtype=str)

In [233]:
df['grade'].cat.categories

Index(['A', 'B', 'C'], dtype='str')

In [234]:
df['grade'].cat.codes

0    0
1    1
2    0
3    2
dtype: int8

##### Operations on categories

In [235]:
# Rename the categories
df['grade'] = df['grade'].cat.rename_categories({
    'A': 'Excellent',
    'B': 'Good',
    'C': 'Medium',
})
df['grade']

0    Excellent
1         Good
2    Excellent
3       Medium
Name: grade, dtype: category
Categories (3, str): ['Excellent', 'Good', 'Medium']

In [236]:
# Set categories
df['grade'] = df['grade'].cat.set_categories(['Excellent', 'Good', 'Medium', "Bad"])
df['grade']

0    Excellent
1         Good
2    Excellent
3       Medium
Name: grade, dtype: category
Categories (4, str): ['Excellent', 'Good', 'Medium', 'Bad']

In [237]:
# Remove categories
df['grade'] = df['grade'].cat.remove_categories('Bad')
print(df['grade'])

# If you deleted the category which exists in dataframe, NaN will be at location of element having the deleted category.

# Remove unused categories
df['grade'] = df['grade'].cat.remove_unused_categories()

0    Excellent
1         Good
2    Excellent
3       Medium
Name: grade, dtype: category
Categories (3, str): ['Excellent', 'Good', 'Medium']


In [238]:
# Ordered categories
df['grade'] = df['grade'].cat.set_categories(['Excellent', 'Good', 'Medium', 'Bad'], ordered=True)
print(df['grade'], '\n')

# You can also compare ordered categoricals.
print(df['grade'][0] < df['grade'][1], '\n')
print(df['grade'] > 'Excellent')

0    Excellent
1         Good
2    Excellent
3       Medium
Name: grade, dtype: category
Categories (4, str): ['Excellent' < 'Good' < 'Medium' < 'Bad'] 

True 

0    False
1     True
2    False
3     True
Name: grade, dtype: bool


##### Sorting

In [239]:
df = df.sort_values(by='grade')
df

,id,grade
0,1,Excellent
2,3,Excellent
1,2,Good
3,4,Medium


##### Adding another categories & Reordering categories order

In [240]:
# You can add new categories. They will be stored at last of categories in ordering.
df['grade'] = df['grade'].cat.add_categories('Very Bad')
print(df['grade'], '\n')

# So we have to reorder to store them in order.
df['grade'] = df['grade'].cat.reorder_categories(['Excellent', 'Good', 'Medium', 'Bad', 'Very Bad'])
print(df['grade'])

0    Excellent
2    Excellent
1         Good
3       Medium
Name: grade, dtype: category
Categories (5, str): ['Excellent' < 'Good' < 'Medium' < 'Bad' < 'Very Bad'] 

0    Excellent
2    Excellent
1         Good
3       Medium
Name: grade, dtype: category
Categories (5, str): ['Excellent' < 'Good' < 'Medium' < 'Bad' < 'Very Bad']


##### Grouping

In [241]:
df.groupby('grade', observed=False).size()

grade
Excellent    2
Good         1
Medium       1
Bad          0
Very Bad     0
dtype: int64